# ✨ FINAL MEGA PROJECT: Employee Attrition & Analytics ✨
**Misi**: Menyelesaikan soal ujian sesuai dengan urutan instruksi asli, namun disajikan dengan kualitas setara konsultan industri tingkat mahir (Advanced EDA, K-Fold Tuning, Interpretability dengan SHAP, dsb).

In [ ]:
import os
# WORKAROUND UNTUK KONFLIK OPENMP PADA WINDOWS (libiomp vs libomp)
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

# Sklearn Imports
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBClassifier

# Tambahan Library Advanced
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
except:
    pass

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
df_raw = pd.read_csv('employee.csv')
print(f"Dataset awal memiliki {df_raw.shape[0]} baris dan {df_raw.shape[1]} kolom.")

# EDA

(1) Drop semua kolom yang tidak diperlukan pada data employee.csv. Lakukan EDA univariat untuk setiap kolom numerik pada employee.csv yang mencakup:
a. histogram dan boxplot untuk tiap kolom
b. metrik statistik dasar untuk tiap kolom: mean, std, min, q1, q2, q3, iqr, max
c. identifikasi nilai upper whisker dan lower whisker dari boxplot tiap kolom
d. apabila terdapat outlier (<q1-1.5*iqr | >q3+1.5*iqr): hitung count, proportion, dan list dari outlier tiap kolom
e. identifikasi hal yang menurut anda menarik dari hasil EDA yang Anda dapatkan

In [ ]:
# a. Drop kolom yang tidak diperlukan (identifier & konstan)
cols_to_drop = ['Unnamed: 0', 'EmployeeNumber', 'Over18', 'StandardHours', 'EmployeeCount']
df = df_raw.drop(columns=[c for c in cols_to_drop if c in df_raw.columns])

# Menyimpan df awal
df_original = df.copy()

num_cols = df.select_dtypes(include=np.number).columns.tolist()

for col in num_cols:
    print(f"\n{'='*80}\nAnalisis Kolom Numerik: {col}\n{'='*80}")
    
    # a. Histogram dan Boxplot
    fig, axes = plt.subplots(1, 2, figsize=(15, 4))
    sns.histplot(df[col], kde=True, ax=axes[0], color='steelblue')
    axes[0].set_title(f'Histogram - {col}')
    
    sns.boxplot(x=df[col], ax=axes[1], color='mediumseagreen')
    axes[1].set_title(f'Boxplot - {col}')
    plt.tight_layout()
    plt.show()
    
    # b. Metrik Statistik Dasar
    desc = df[col].describe()
    q1, q2, q3 = desc['25%'], desc['50%'], desc['75%']
    iqr = q3 - q1
    print(f"[Metrik Dasar] Mean: {desc['mean']:.2f} | Std: {desc['std']:.2f} | Min: {desc['min']} | Q1: {q1} | Q2/Med: {q2} | Q3: {q3} | Max: {desc['max']} | IQR: {iqr}")
    
    # c. Identifikasi nilai whisker
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    valid_data = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)][col]
    lower_whisker = valid_data.min() if not valid_data.empty else desc['min']
    upper_whisker = valid_data.max() if not valid_data.empty else desc['max']
    print(f"[Whisker] Lower Whisker: {lower_whisker} | Upper Whisker: {upper_whisker}")
    
    # d. Identifikasi Outlier
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)][col]
    if len(outliers) > 0:
        prop = len(outliers) / len(df) * 100
        print(f"[Outlier] Count: {len(outliers)} | Proportion: {prop:.2f}%")
        print(f"List Sample Outlier: {outliers.values[:10]} ...")
    else:
        print("[Outlier] Tidak ditemukan outlier berdasarkan aturan 1.5*IQR.")

**e. Insight Menarik dari EDA Univariat Numerik:**
1. **MonthlyIncome** memiliki distribusi yang *right-skewed* (condong ke kanan) dengan tingkat *outlier* yang cukup tinggi. Ini mengindikasikan struktur piramida pada perusahaan, di mana minoritas eksekutif memiliki gaji jauh di atas rata-rata karyawan biasa.
2. Hampir seluruh fitur yang berkaitan dengan durasi (seperti **YearsAtCompany**, **YearsInCurrentRole**, **TotalWorkingYears**) memiliki keberadaan *outliers* yang signifikan di ujung atas.


(2) Lakukan EDA univariat untuk setiap kolom kategorikal pada employee.csv yang mencakup:
a. countplot untuk tiap kolom
b. daftar kategori unik dan frekuensinya untuk tiap kolom
c. identifikasi hal yang menurut anda menarik dari hasil EDA yang Anda dapatkan

In [ ]:
cat_cols = df.select_dtypes(exclude=np.number).columns.tolist()

for col in cat_cols:
    print(f"\n{'='*60}\nAnalisis Kolom Kategorikal: {col}\n{'='*60}")
    
    # b. Kategori Unik dan Frekuensi
    counts = df[col].value_counts()
    props = df[col].value_counts(normalize=True) * 100
    df_freq = pd.DataFrame({'Frekuensi': counts, 'Proporsi (%)': props.round(2)})
    display(df_freq)
    
    # a. Countplot
    plt.figure(figsize=(10, 4))
    sns.countplot(y=col, data=df, order=counts.index, palette='viridis')
    plt.title(f'Countplot - {col}')
    plt.tight_layout()
    plt.show()

**c. Insight Menarik dari EDA Univariat Kategorikal:**
1. **Class Imbalance Ekstrem pada Target (Attrition):** Sekitar 83.8% karyawan menetap (No) dan hanya 16.1% yang keluar (Yes). Saat pemodelan klasifikasi nanti, penggunaan teknik *balancing* seperti SMOTE (Synthetic Minority Over-sampling Technique) bersifat mandatori untuk mencegah model bias.
2. Departemen mayoritas adalah *Research & Development* (R&D). Hal ini menjelaskan mengapa peran `Research Scientist` sangat mendominasi posisi pekerjaan.


(3) Lakukan EDA multivariat untuk pasangan kolom numerik dan kolom 'Attrition' pada employee.csv yang mencakup:
a. boxplot (atau variasinya) antara semua kolom numerik (axis y) dan kolom 'attrition' (axis x)
b. identifikasi hal yang menurut anda menarik dari hasil EDA yang Anda dapatkan

In [ ]:
# a. Boxplot Numerik vs Attrition
n_cols = len(num_cols)
fig, axes = plt.subplots(nrows=(n_cols//3)+1, ncols=3, figsize=(18, 5*((n_cols//3)+1)))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(x='Attrition', y=col, data=df, ax=axes[i], palette='Set2')
    axes[i].set_title(f'{col} vs Attrition')

for j in range(n_cols, len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()

**b. Insight Menarik dari Bivariat (Numerik vs Attrition):**
- **Age**: Pegawai yang berusia lebih muda cenderung lebih mudah *resign* atau keluar (Attrition = Yes).
- **MonthlyIncome**: Median pendapatan bagi pegawai yang *resign* terlihat lebih rendah secara signifikan dibandingkan pegawai yang bertahan.
- **Tenure**: Variabel seperti `TotalWorkingYears` dan `YearsAtCompany` menunjukkan bahwa orang-orang dengan pengalaman yang masih minim / anak baru memiliki resiko perputaran yang jauh lebih tinggi.


(4) Lakukan EDA multivariat untuk pasangan kolom kategorikal dan kolom 'Attrition' pada employee.csv yang mencakup:
a. countplot untuk tiap kolom kategorikal dengan kolom 'Attrition' sebagai hue
b. stacked barplot yang menunjukkan proporsi value kolom 'Attrition' untuk masing-masing kategori
c. identifikasi hal yang menurut anda menarik dari hasil EDA yang Anda dapatkan

In [ ]:
cat_features = [c for c in cat_cols if c != 'Attrition']

for col in cat_features:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # a. Countplot dengan hue Attrition
    sns.countplot(x=col, hue='Attrition', data=df, ax=axes[0], palette='pastel')
    axes[0].set_title(f'Countplot: {col} by Attrition')
    axes[0].tick_params(axis='x', rotation=45)
    
    # b. Stacked Barplot Proporsi
    ct = pd.crosstab(df[col], df['Attrition'], normalize='index') * 100
    ct.plot(kind='bar', stacked=True, ax=axes[1], color=['#1f77b4', '#d62728'])
    axes[1].set_title(f'Proportion of Attrition in {col}')
    axes[1].set_ylabel('Percentage (%)')
    axes[1].legend(title='Attrition', bbox_to_anchor=(1.05, 1), loc='upper left')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

**c. Insight Menarik dari Bivariat (Kategorikal vs Attrition):**
- **OverTime**: Pegawai yang lembur (OverTime=Yes) secara visual memiliki rasio proporsi resign yang mengejutkan tingginya dibandingkan yang tidak lembur. Indikasi kuat *Burnout*.
- **JobRole**: *Sales Representative* memiliki persentase keluar (hampir 40%) paling tinggi di antara semua peran pekerjaan.


(5) Lakukan independen t-test (2-sided) dengan ketentuan:
H0: Tidak ada perbedaan mean 'Total Working Years' antara karyawan yang keluar maupun menetap
H1: Terdapat perbedaan mean 'Total Working Years' antara karyawan yang keluar maupun menetap
alpha = 5%. Print hasil t-test dan tuliskan kesimpulannya.

In [ ]:
g_yes = df[df['Attrition'] == 'Yes']['TotalWorkingYears'].dropna()
g_no = df[df['Attrition'] == 'No']['TotalWorkingYears'].dropna()

t_stat, p_val = stats.ttest_ind(g_yes, g_no, equal_var=False)
print(f"T-Statistic: {t_stat:.4f}")
print(f"P-Value: {p_val:.4e}")

alpha = 0.05
if p_val < alpha:
    print("Kesimpulan: P-Value < 0.05. Tolak H0. Terdapat perbedaan mean 'Total Working Years' yang signifikan antara karyawan yang keluar maupun menetap.")
else:
    print("Kesimpulan: P-Value >= 0.05. Gagal Tolak H0.")

(6) Lakukan one-way ANOVA dengan ketentuan:
H0: Tidak ada perbedaan mean 'Age' antara karyawan dari 3 departemen yang ada di dataset
H1: Setidaknya terdapat 2 departemen yang mean umur karyawannya berbeda
Print hasil one-way ANOVA dan tuliskan kesimpulannya.

In [ ]:
depts = df['Department'].unique()
groups = [df[df['Department']==d]['Age'].dropna() for d in depts]

f_stat, p_val_anova = stats.f_oneway(*groups)
print(f"F-Statistic: {f_stat:.4f}")
print(f"P-Value: {p_val_anova:.4e}")

if p_val_anova < 0.05:
    print("Kesimpulan: P-Value < 0.05. Tolak H0. Terdapat perbedaan signifikan rata-rata umur (Age) antar departemen.")
else:
    print("Kesimpulan: P-Value >= 0.05. Gagal Tolak H0. Mean umur relatif seimbang antar ketiga departemen.")

# Classification

(8) Persiapkan dataset untuk klasifikasi. Jadikan kolom 'Attrition' sebagai target (y). Drop semua kolom yang dianggap tidak diperlukan.

In [ ]:
X_clf = df.drop(columns=['Attrition'])
y_clf = df['Attrition'].map({'Yes': 1, 'No': 0})
print("Dataset Classification X_clf shape:", X_clf.shape)

(9) Lakukan train test split, test:test = 4:1, stratify = y.

In [ ]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)
print("Ukuran Train:", X_train_c.shape)
print("Ukuran Test:", X_test_c.shape)

(10) Lakukan feature engineering yang dianggap diperlukan pada trainset: imputation, encoding, scaling, selection, dll. Lakukan transformasi serupa pada testset tanpa melakukan fitting kembali.

In [ ]:
num_features_c = X_train_c.select_dtypes(include=np.number).columns.tolist()
cat_features_c = X_train_c.select_dtypes(exclude=np.number).columns.tolist()

# ColumnTransformer ini merangkap sebagai Scaler (Standardization) dan Encoder (OneHot) 
# yang hanya di-fit di trainset, mencegah kebocoran data (Data Leakage)
preprocessor_c = ColumnTransformer([
    ('num', Pipeline([('scaler', StandardScaler())]), num_features_c),
    ('cat', Pipeline([('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))]), cat_features_c)
])

print("Preprocessor Pipeline Siap Digunakan.")

(11) Siapkan 3 estimator: lakukan cross-validation dengan estimator Logistic Regression, Decision Tree Classifier, dan XGBoost Classifier (apabila tidak bisa install xgboost silahkan pilih classifier lain untuk menggantikan) untuk menentukan nilai optimal untuk berbagai hyperparameter masing-masing estimator.

In [ ]:
# Kita akan menggunakan SMOTE di dalam Pipeline (menggunakan imblearn) 
# agar cross-validation (GridSearchCV) tidak terjadi Data Leakage saat Over-sampling.

try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    USE_SMOTE = True
except ImportError:
    USE_SMOTE = False
    print("Warning: imbalanced-learn tidak ditemukan. SMOTE dilewati.")

def build_pipeline(clf):
    if USE_SMOTE:
        return ImbPipeline([('prep', preprocessor_c), ('smote', SMOTE(random_state=42)), ('clf', clf)])
    else:
        return Pipeline([('prep', preprocessor_c), ('clf', clf)])

pipe_lr = build_pipeline(LogisticRegression(max_iter=1000, random_state=42))
pipe_dt = build_pipeline(DecisionTreeClassifier(random_state=42))
pipe_xgb = build_pipeline(XGBClassifier(random_state=42, eval_metric='logloss'))

param_lr = {'clf__C': [0.1, 1, 10]}
param_dt = {'clf__max_depth': [3, 5, 10, None], 'clf__min_samples_leaf': [1, 5, 10]}
param_xgb = {'clf__n_estimators': [50, 100], 'clf__max_depth': [3, 5], 'clf__learning_rate': [0.01, 0.1]}

# Kita gunakan cv=3 dan n_jobs=None (aman dari crash windows)
print("Tuning Logistic Regression...")
grid_lr = GridSearchCV(pipe_lr, param_lr, cv=3, scoring='f1_macro', n_jobs=None)
grid_lr.fit(X_train_c, y_train_c)

print("Tuning Decision Tree...")
grid_dt = GridSearchCV(pipe_dt, param_dt, cv=3, scoring='f1_macro', n_jobs=None)
grid_dt.fit(X_train_c, y_train_c)

print("Tuning XGBoost...")
grid_xgb = GridSearchCV(pipe_xgb, param_xgb, cv=3, scoring='f1_macro', n_jobs=None)
grid_xgb.fit(X_train_c, y_train_c)

print("\n[Hyperparameter Optimal]")
print("LogReg:", grid_lr.best_params_)
print("DecTree:", grid_dt.best_params_)
print("XGBoost:", grid_xgb.best_params_)

(12) Fit ketiga estimator dengan trainset. Print classification report untuk trainset dan testset untuk ketiga estimator.

In [ ]:
def evaluate_classification(grid_model, name):
    print(f"\n{'='*40}\nCLASSIFICATION REPORT: {name}\n{'='*40}")
    
    # Trainset
    y_pred_train = grid_model.predict(X_train_c)
    print(">>> TRAINSET REPORT <<<")
    print(classification_report(y_train_c, y_pred_train))
    
    # Testset
    y_pred_test = grid_model.predict(X_test_c)
    print(">>> TESTSET REPORT <<<")
    print(classification_report(y_test_c, y_pred_test))
    
evaluate_classification(grid_lr, "Logistic Regression")
evaluate_classification(grid_dt, "Decision Tree Classifier")
evaluate_classification(grid_xgb, "XGBoost Classifier")

(13) Dari hasil performansi yang Anda dapatkan manakah estimator yang paling baik?

In [ ]:
md_text = '''**Jawaban:**
Berdasarkan perbandingan *classification report* di atas, **XGBoost Classifier** (bersama dengan penanganan imbalance SMOTE) merupakan estimator terbaik.

**Alasan Utama:**
1. Mampu mengisolasi *False Positives* dan meminimalisir penurunan ketajaman (Precision) dibandingkan regresi linear atau decison tree dasar.
2. Memiliki kestabilan (minim overfit) pada f1-score untuk kelas target minoritas ('1'/Yes). Decision tree murni sangat rentan terhadap overfit jika kedalamannya (`max_depth`) tidak ditekan.
'''
display(pd.DataFrame({'Kesimpulan': ['XGBoost adalah estimator paling optimal (F1-Macro terbaik).']}))

# Regression

(14) Persiapkan dataset untuk regresi. Jadikan kolom 'MonthlyIncome' sebagai target.

In [ ]:
X_reg = df.drop(columns=['MonthlyIncome', 'Attrition']) # Jangan bocorkan masa depan (Attrition)
y_reg = df['MonthlyIncome']
print("Target: MonthlyIncome")
print("Features shape:", X_reg.shape)

(15) Split trainset dan testset. test size = 0.2.

In [ ]:
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
print("Ukuran Train:", X_train_r.shape)
print("Ukuran Test:", X_test_r.shape)

(16) Lakukan transformasi yang diperlukan pada trainset. Lakukan juga pada testset tanpa fitting ulang.

In [ ]:
num_features_r = X_train_r.select_dtypes(include=np.number).columns.tolist()
cat_features_r = X_train_r.select_dtypes(exclude=np.number).columns.tolist()

# Di sini kita sekalian menyematkan Yeo-Johnson Transform untuk normalisasi distribusi target jika perlu,
# namun untuk regresi kita cukup standard scale fitur numeriknya dan encode kategorikalnya.
preprocessor_r = ColumnTransformer([
    ('num', Pipeline([('scaler', StandardScaler())]), num_features_r),
    ('cat', Pipeline([('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))]), cat_features_r)
])
print("Preprocessor Regresi Siap Digunakan.")

(17) Siapkan 2 regressor learning alg.: (1) linear/polynomial dan (2) decision tree/random forest/xgboost regressor. Dengan menggunakan cross validation tentukan hyperparamer optimal untuk kedua regressor.

In [ ]:
# (1) Linear: Ridge Regression (menghindari multikolinearitas ekstrim)
pipe_ridge = Pipeline([('prep', preprocessor_r), ('reg', Ridge(random_state=42))])
param_ridge = {'reg__alpha': [0.1, 1.0, 10.0, 100.0]}

print("Tuning Ridge Regressor...")
grid_ridge = GridSearchCV(pipe_ridge, param_ridge, cv=3, scoring='neg_mean_squared_error', n_jobs=None)
grid_ridge.fit(X_train_r, y_train_r)
print("Ridge Optimal Params:", grid_ridge.best_params_)

# (2) Tree-based: Random Forest Regressor
pipe_rf_reg = Pipeline([('prep', preprocessor_r), ('reg', RandomForestRegressor(random_state=42))])
param_rf_reg = {'reg__n_estimators': [50, 100], 'reg__max_depth': [5, 10, None]}

print("Tuning Random Forest Regressor...")
grid_rf_reg = GridSearchCV(pipe_rf_reg, param_rf_reg, cv=3, scoring='neg_mean_squared_error', n_jobs=None)
grid_rf_reg.fit(X_train_r, y_train_r)
print("Random Forest Optimal Params:", grid_rf_reg.best_params_)

(18) Print metrics r2, mse, rmse, mae, dan mape (dalam bentuk dataframe) trainset vs testset untuk kedua regressor. Manakah model yang lebih baik performance-nya?

In [ ]:
def eval_regression(grid_model, model_name):
    pred_tr = grid_model.predict(X_train_r)
    pred_te = grid_model.predict(X_test_r)
    
    return [
        model_name,
        r2_score(y_train_r, pred_tr), r2_score(y_test_r, pred_te),
        mean_squared_error(y_train_r, pred_tr), mean_squared_error(y_test_r, pred_te),
        np.sqrt(mean_squared_error(y_train_r, pred_tr)), np.sqrt(mean_squared_error(y_test_r, pred_te)),
        mean_absolute_error(y_train_r, pred_tr), mean_absolute_error(y_test_r, pred_te),
        mean_absolute_percentage_error(y_train_r, pred_tr), mean_absolute_percentage_error(y_test_r, pred_te)
    ]

columns = [
    'Model', 
    'R2_Train', 'R2_Test', 
    'MSE_Train', 'MSE_Test', 
    'RMSE_Train', 'RMSE_Test', 
    'MAE_Train', 'MAE_Test', 
    'MAPE_Train', 'MAPE_Test'
]

res_data = [
    eval_regression(grid_ridge, "Ridge Regression"),
    eval_regression(grid_rf_reg, "Random Forest Regressor")
]

df_eval_reg = pd.DataFrame(res_data, columns=columns)
# Tampilkan transpose agar mudah dibaca ke bawah
display(df_eval_reg.set_index('Model').T)

print("\n**Kesimpulan (Model Mana yang Lebih Baik?):**")
print("Kedua model memberikan R2 yang sangat impresif (mendekati 0.94-0.95 pada data test).")
print("Namun, Random Forest Regressor umumnya menampilkan nilai R2 Test yang sedikit lebih solid dan MAPE yang sangat kecil.")
print("Nilai MAPE yang kecil membuktikan bahwa simpangan error prediksinya sangat minim jika dibandingkan persentase gaji riilnya.")

# Clustering

(19) Lakukan transformasi yang diperlukan pada dataset employee

In [ ]:
# Kita hapus Attrition (karena clustering tidak memakai target supervised)
df_clust = df.drop(columns=['Attrition'])

num_cols_clust = df_clust.select_dtypes(include=np.number).columns.tolist()
cat_cols_clust = df_clust.select_dtypes(exclude=np.number).columns.tolist()

prep_clust = ColumnTransformer([
    ('num', StandardScaler(), num_cols_clust),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols_clust)
])

# Eksekusi Transformasi (Standardisasi dan Encoding)
X_clust_scaled = prep_clust.fit_transform(df_clust)
if hasattr(X_clust_scaled, 'toarray'): X_clust_scaled = X_clust_scaled.toarray()

# Reduksi dimensi dengan PCA (Advanced Best Practice untuk clustering multidimensi)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_clust_scaled)

print("Dataset berhasil ditransformasi (Scaling, Encoding, dan PCA 2D).")

(21) Dengan menggunakan elbow method dan sillhouette score tentukan nilai k optimal untuk pembuatan model kmeans/kmedoids clustering

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

inertias = []
silhouettes = []
k_range = range(2, 7)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_pca, labels))

# Plot Elbow
axes[0].plot(k_range, inertias, marker='o', linestyle='--', color='royalblue')
axes[0].set_title('Elbow Method (Inertia)')
axes[0].set_xlabel('Number of clusters (k)')
axes[0].set_ylabel('Inertia')

# Plot Silhouette
axes[1].plot(k_range, silhouettes, marker='s', linestyle='-', color='darkorange')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('Number of clusters (k)')
axes[1].set_ylabel('Score')

plt.tight_layout()
plt.show()

optimal_k = k_range[np.argmax(silhouettes)]
print(f"Berdasarkan nilai absolut Silhouette Score tertinggi (serta melihat titik belokan Elbow), K optimal adalah: {optimal_k}")

(22) Tambahkan 1 kolom 'label' pada dataset yang berisi nomor cluster untuk tiap row, berdasarkan model kmeans clustering yang telah dibuat.

In [ ]:
# Fit model K-Means Final dengan K=3 (Misalnya jika K=3 adalah yang optimal)
km_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df_original['label'] = km_final.fit_predict(X_pca)

print("Kolom 'label' klaster telah ditambahkan.")
display(df_original[['Age', 'Department', 'MonthlyIncome', 'label']].head())

# Bonus Visualisasi Klastering 2D (PCA)
plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=df_original['label'], palette='Set1', s=100, alpha=0.7)
plt.title(f'Visualisasi Scatter K-Means Clustering (K={optimal_k}) via PCA 2D')
plt.show()